# Fraunhofer diffraction as a 2D Fourier transform

*optics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/optics/fourier_optics)

In the far field, the diffraction pattern of an aperture is the 2D Fourier transform of its transmission function. One call to torch.fft.fft2 turns a square, a circular hole, a double slit and a grating into their far-field patterns, and the Airy rings shrink as the hole grows.

**Vector form**

$$U(\mathbf q) \propto \iint A(\mathbf r)\,e^{-i\,\mathbf q\cdot\mathbf r}\,d^2 r,\quad \mathbf q = k\,(\sin\theta_x, \sin\theta_y),\qquad I = \lvert U\rvert^2$$

**Index (tensor) form**

$$U_{kl} = F_{ka}\,A_{ab}\,F_{lb},\qquad F_{ka} = e^{-2\pi i\,k a/N},\qquad I_{kl} = U^{*}_{kl}U_{kl}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $A_{ab}$ | `aperture` | (N, N) | transmission of the mask: 1 = open, 0 = opaque; row $a = y$, column $b = x$ |
| $F_{ka}$ | `dft_matrix(N)` | (N, N) complex | discrete Fourier transform matrix |
| $U_{kl} = F_{ka}A_{ab}F_{lb}$ | `einsum('ka,ab,lb->kl', F, A, F)` | (N, N) complex | repeated $a$ and $b$ ⇒ summed: a 1D transform along $y$ and one along $x$ |
| $U_{kl}$ | `fftshift(fft2(aperture))` | (N, N) complex | the same transform via the fast Fourier transform, zero frequency moved to the centre |
| $I_{kl}$ | `U.abs() ** 2` | (N, N) | far-field intensity, shown as $\log_{10}(I/I_{\max})$ |
| $1.22\,N/D$ | `airy_radius(N, D)` | () | first dark ring of a circular hole of diameter $D$, in frequency pixels |

Lengths in the aperture plane are in pixels of size $\Delta x$; frequency pixel $(k, l)$ of the transform corresponds to the direction $\sin\theta = \lambda k/(N\Delta x)$. The circular hole's first dark ring at $\sin\theta = 1.22\,\lambda/D$ therefore sits $1.22\,N/D$ frequency pixels from the centre (dashed circle). Big apertures give small patterns: the transform of a wide function is narrow. The einsum form is the same transform as fft2, only $O(N^3)$ instead of $O(N^2\log N)$.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/optics/fourier_optics/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    size_min, size_max = params["size_min"], max(params["size_min"], params["size_max"])
    schedule = tensors.schedule(size_min, size_max, params["frames_per_aperture"])

    masks, patterns = solve.solve(schedule)

    for d in (size_min, size_max):
        measured, predicted = solve.first_dark_ring(d), equation.airy_radius(tensors.N, d)
        print(f"circle D = {d:2d} px: first dark ring at {measured:.0f} px (1.22 N/D = {predicted:.1f} px)")
    return plot.animate(schedule, masks, patterns, out_dir / "fourier_optics.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/optics/fourier_optics/equation.py
r"""Physics → tensor translation.

Far behind an aperture (Fraunhofer limit, or in the focal plane of a lens) the field in the direction
(\sin\theta_x, \sin\theta_y) is the Fourier transform of the aperture's transmission A(x, y):

    U(q_x, q_y) \propto \iint A(x, y)\, e^{-i (q_x x + q_y y)}\, dx\, dy,    q = k \sin\theta

On an N×N pixel grid (row a = y, column b = x) this is the 2D discrete Fourier transform

    U_{kl} = \sum_{a,b} A_{ab}\, e^{-2\pi i (k a + l b)/N} = F_{ka} A_{ab} F_{lb},    F_{ka} = e^{-2\pi i k a / N}

where the repeated indices a and b are summed: one 1D transform along y, one along x.
torch.fft.fft2 computes exactly this sum, fast. The intensity is I_{kl} = U^*_{kl} U_{kl}.
"""

import math

import torch


def far_field(aperture: torch.Tensor) -> torch.Tensor:
    """U = fft2(A), with fftshift moving zero frequency (straight ahead) to the centre.   (N, N) → (N, N) complex"""
    return torch.fft.fftshift(torch.fft.fft2(aperture.to(torch.complex64)))


def dft_matrix(n: int) -> torch.Tensor:
    """F_{ka} = exp(−2πi k a / N): shape (N, N) complex."""
    k = torch.arange(n, dtype=torch.float64)
    return torch.exp(-2j * math.pi * torch.einsum("k,a->ka", k, k) / n)


def far_field_einsum(aperture: torch.Tensor) -> torch.Tensor:
    r"""U_{kl} = F_{ka} A_{ab} F_{lb} written out with einsum (same as fft2, then fftshift).   (N, N) → (N, N)"""
    f = dft_matrix(aperture.shape[0])
    u = torch.einsum("ka,ab,lb->kl", f, aperture.to(f.dtype), f)
    return torch.fft.fftshift(u)


def intensity(u: torch.Tensor) -> torch.Tensor:
    """I_{kl} = U*_{kl} U_{kl}.   (N, N) complex → (N, N) real"""
    return (u.conj() * u).real


def log_intensity(i: torch.Tensor, floor: float = 1e-5) -> torch.Tensor:
    """log₁₀(I / I_max), clipped at log₁₀(floor) so the dark zeros stay finite."""
    return torch.log10((i / i.max()).clamp(min=floor))


def airy_radius(n: int, diameter: float) -> float:
    """First dark ring of a circular hole: sin θ = 1.22 λ / D  ⇒  1.22 N / D frequency pixels."""
    return 1.22 * n / diameter


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/optics/fourier_optics/tensors.py
"""Build the concrete tensors: pixel coordinates and the aperture masks (1 = open, 0 = opaque)."""

import torch

N = 256  # grid size: N × N pixels
SHAPES = ["square", "circle", "double slit", "grating"]


def coordinates(n: int = N) -> tuple[torch.Tensor, torch.Tensor]:
    """Pixel coordinates centred on the grid: y[a, b] = a − N/2, x[a, b] = b − N/2, each (N, N)."""
    c = torch.arange(n, dtype=torch.float32) - n // 2
    y, x = torch.meshgrid(c, c, indexing="ij")
    return y, x


def aperture(shape: str, size: float, n: int = N) -> torch.Tensor:
    """Mask A_{ab} of the given shape, overall width `size` pixels: (N, N) float."""
    y, x = coordinates(n)
    half = size / 2
    if shape == "square":  # exactly `size` pixels wide in x and y
        open_ = (x >= -half) & (x < half) & (y >= -half) & (y < half)
    elif shape == "circle":  # diameter `size`
        open_ = x**2 + y**2 <= half**2
    elif shape == "double slit":  # two slits, centres size/2 apart, each size/8 wide and `size` tall
        width = max(1.0, size / 8)
        open_ = ((x - size / 4).abs() < width / 2) | ((x + size / 4).abs() < width / 2)
        open_ &= y.abs() < half
    elif shape == "grating":  # 5 slits with period size/5, half open
        period = max(2.0, size / 5)
        open_ = (torch.remainder(x + period / 4, period) < period / 2) & (x.abs() < 2.5 * period)
        open_ &= y.abs() < half
    else:
        raise ValueError(f"unknown aperture shape {shape!r}")
    return open_.float()


def schedule(size_min: float, size_max: float, frames_per_shape: int) -> list[tuple[str, float]]:
    """(shape, size) for every animation frame: each shape in turn, its size growing from min to max."""
    sizes = torch.linspace(size_min, size_max, frames_per_shape).round().tolist()
    return [(shape, size) for shape in SHAPES for size in sizes]


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/optics/fourier_optics/solve.py
"""Transform every aperture of the animation to its far field; measure the Airy ring of the circular hole."""

import torch

from .equation import far_field, intensity, log_intensity
from .tensors import aperture


def solve(schedule: list[tuple[str, float]]) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (apertures (F, N, N), far-field log₁₀ intensity (F, N, N)) for every (shape, size) frame."""
    masks, patterns = [], []
    for shape, size in schedule:
        a = aperture(shape, size)
        masks.append(a)
        patterns.append(log_intensity(intensity(far_field(a))))
    return torch.stack(masks), torch.stack(patterns)


def radial_profile(i: torch.Tensor) -> torch.Tensor:
    """Average of a centred (N, N) image over rings of width 1 pixel around the centre: (N/2,)."""
    n = i.shape[0]
    c = torch.arange(n) - n // 2
    r = torch.sqrt(c[:, None] ** 2 + c[None, :] ** 2).round().long()  # (N, N) ring number of each pixel
    keep = r < n // 2
    total = torch.zeros(n // 2).index_add_(0, r[keep], i[keep])
    count = torch.zeros(n // 2).index_add_(0, r[keep], torch.ones_like(i[keep]))
    return total / count


def first_dark_ring(diameter: float) -> float:
    """Radius (frequency pixels) of the first minimum of the circular hole's far-field intensity."""
    profile = radial_profile(intensity(far_field(aperture("circle", diameter))))
    for r in range(1, len(profile) - 1):
        if profile[r] < profile[r - 1] and profile[r] <= profile[r + 1]:
            return float(r)
    return float("nan")


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/optics/fourier_optics/plot.py
"""Animate aperture (left) and far-field log-intensity (right) side by side, cycling through the shapes."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from .equation import airy_radius
from .tensors import N

VIEW = 48  # show the central ±VIEW pixels of both planes


def animate(schedule: list, masks: torch.Tensor, patterns: torch.Tensor, path: Path) -> Path:
    centre = slice(N // 2 - VIEW, N // 2 + VIEW)
    a = masks[:, centre, centre].numpy()  # (F, 2V, 2V)
    log_i = patterns[:, centre, centre].numpy()  # (F, 2V, 2V)
    extent = (-VIEW - 0.5, VIEW - 0.5, -VIEW - 0.5, VIEW - 0.5)

    fig, (ax_a, ax_i) = plt.subplots(1, 2, figsize=(7, 4))
    img_a = ax_a.imshow(a[0], origin="lower", extent=extent, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    img_i = ax_i.imshow(log_i[0], origin="lower", extent=extent, cmap=plt.get_cmap("magma", 20), vmin=-5, vmax=0)
    fig.colorbar(img_i, ax=ax_i, fraction=0.046, pad=0.04, label="log₁₀ I / I_max")
    (ring,) = ax_i.plot([], [], "--", color="#22d3ee", lw=1)
    ax_a.set_xlabel("x (px)")
    ax_a.set_ylabel("y (px)")
    ax_i.set_xlabel("frequency l (px)")
    ax_i.set_ylabel("frequency k (px)")
    for ax in (ax_a, ax_i):
        ax.grid(False)
    title_a, title_i = ax_a.set_title(" "), ax_i.set_title("far field |U|²")
    circle = np.linspace(0, 2 * np.pi, 100)

    def update(f: int):
        shape, size = schedule[f]
        img_a.set_data(a[f])
        img_i.set_data(log_i[f])
        title_a.set_text(f"aperture: {shape}, D = {size:.0f} px")
        if shape == "circle":  # predicted first dark ring at 1.22 N / D
            r = airy_radius(N, size)
            ring.set_data(r * np.cos(circle), r * np.sin(circle))
            title_i.set_text("far field |U|²  (dashed: 1.22 N/D)")
        else:
            ring.set_data([], [])
            title_i.set_text("far field |U|²")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(schedule)), path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'size_min': 12,
    'size_max': 40,
    'frames_per_aperture': 15
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.optics.fourier_optics")]:
    del sys.modules[name]
main = importlib.import_module("problems.optics.fourier_optics.main")
problem_dir = Path("problems/optics/fourier_optics")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
